# A Vulnerability Prioritization and Exposure Management (VPEM) Example Use Case

We previously loaded the NVD CVE data into Neo4j. Together with some sample infrastructure data, we can now demonstrate how to use this data for simple but illustrative vulnerability prioritization and exposure management (VPEM) use cases.

In [8]:
from dotenv import load_dotenv
import os
from neo4j import GraphDatabase

load_dotenv()

# Connection details
URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
AUTH = (os.getenv("NEO4J_USER", "neo4j"), os.getenv("NEO4J_PASSWORD", "password"))
DB = os.getenv("NEO4J_DB", "nvd")

# A helper method to run Cypher queries
def run_cypher(query, parameters=None):
    driver = GraphDatabase.driver(URI, auth=AUTH)
    try:
        with driver.session(database=DB) as session:
            result = session.run(query, parameters or {})
            # .data() converts the stream into a list of dictionaries
            return result.data() 
    finally:
        driver.close()

### Identifying Reachable Critical Vulnerabilities

The following code snippet performs a **Reachability Analysis**, which is a cornerstone of modern Vulnerability Prioritization and Exposure Management (VPEM). Instead of looking at a vulnerability in a vacuum, it traces the path from a known threat to a specific point of entry in your infrastructure.

#### What this Query is Doing

The Cypher query traverses your security knowledge graph to connect disparate layers of your technology stack:

1. **Supply Chain Layer:** It starts with a specific, high-risk vulnerability (**CVE-2021-44228**, famously known as *Log4Shell*) and finds the software **Library** that contains it.
2. **Application Layer:** It identifies which **Build Artifacts** (compiled code/images) use that library and which **Applications** are currently running that code.
3. **Infrastructure Layer:** It maps those applications to the specific **Compute Instances** (servers or VMs) where they are hosted.
4. **Exposure Filter:** This is the most critical part. The `EXISTS` clause checks for a direct connection from an internet-facing **Endpoint**. It essentially asks: *"Is there a public URL that resolves to this specific vulnerable server?"*

#### Why This is Important

In a typical enterprise environment, a single "Critical" CVE like Log4Shell might be detected on thousands of assets. Security teams cannot patch everything at once. This use case provides **three essential benefits**:

* **Eliminating "Noise":** Most vulnerability scanners only tell you that a library exists on a disk. This query tells you that the library is **active, running, and exposed to the public internet**. A vulnerability on an internet-facing server is a "Day 0" priority, whereas the same vulnerability on an isolated back-office server can wait.
* **Validating the Attack Path:** It confirms the **exploitability** of the vulnerability. For an attacker to trigger Log4Shell, they usually need to send a malicious string to a network-accessible service. This query identifies exactly where that "front door" is open.
* **Informing Immediate Response:** By returning the `app.name` and `public_ip`, it gives Incident Response teams the exact "who" and "where" needed to take action—such as placing the server behind a Web Application Firewall (WAF) or isolating the subnet while a patch is prepared.

In [9]:
query = """
MATCH (v:CVE)-[:IDENTIFIED_IN]->(l:Library)-[:DEPENDENCY_OF]->(ba:BuildArtifact)-[:RUNNING_AS]->(app:Application)-[:HOSTED_ON]->(i:ComputeInstance)
WHERE (v.id = 'CVE-2021-44228') AND EXISTS { (end:Endpoint)-[:RESOLVES_TO]->(i) }
RETURN app.name, i.public_ip, v.id
"""
results = run_cypher(query)
for record in results:
    print(f"Application: {record['app.name']}, Public IP: {record['i.public_ip']}, CVE: {record['v.id']}")

Application: CustomerFacingAPI, Public IP: 34.201.1.5, CVE: CVE-2021-44228


### Explainer: Blast Radius and Impact Analysis

The following code snippet performs a **Blast Radius Analysis**. While the previous use case focused on how an attacker might *enter* the environment (**Reachability**), this use case focuses on the **Impact**—quantifying exactly what is at risk if a specific application is compromised.

#### What this Query is Doing

This query traverses the graph to bridge the gap between a **software vulnerability** and **cloud permission structures**:

1. **Identity Linkage:** It identifies the **Identity** (Service Account or IAM Role) that the vulnerable **Application** is authorized to use.
2. **Permission Mapping:** It traverses the relationship between that Identity and its attached **IAM Policies**.
3. **Resource Discovery:** It follows those policies to the final **Cloud Service** (e.g., an S3 bucket, a DynamoDB table, or a Key Vault) that the application can actually "touch."

#### Why This is Important

In modern cloud environments, vulnerabilities are rarely the "end game." They are usually the first step in a **Data Breach**. This analysis is vital for several reasons:

* **Quantifying Business Risk:** A "Critical" vulnerability on a web server is bad. A "Critical" vulnerability on a web server that has `FullAccess` to a **Customer PII Bucket** is a catastrophic business risk. This query allows you to categorize vulnerabilities by the value of the data they can access.
* **Enforcing the Principle of Least Privilege:** This query often reveals "Over-privileged" identities. You might discover an application only needs to *read* a specific file, but its assigned policy allows it to *delete* the entire database. Identifying these "Identity-based attack paths" is a key part of Exposure Management.
* **Prioritizing "Crown Jewels":** Organizations often have thousands of vulnerabilities. By focusing on the "Blast Radius," security teams can prioritize patching the applications that serve as the keys to the kingdom—their **Crown Jewel** assets.

In [10]:
query = """
MATCH (v:CVE)-[:IDENTIFIED_IN]->(l:Library)-[:DEPENDENCY_OF]->(ba:BuildArtifact)-[:RUNNING_AS]->(app:Application)
MATCH (app)-[:AUTHENTICATES_VIA]->(id:Identity)-[:ASSUMES]->(pol:IAMPolicy)-[:HAS_ACCESS_TO]->(cs:CloudService)
RETURN v.id, app.name, cs.resource_name, pol.name
"""
results = run_cypher(query)
for record in results:
    print(f"CVE: {record['v.id']}, Application: {record['app.name']}, Resource: {record['cs.resource_name']}, Policy: {record['pol.name']}")

CVE: CVE-2021-44228, Application: CustomerFacingAPI, Resource: acme-customer-pii-data, Policy: DataLakeFullAccess
CVE: CVE-2021-45046, Application: CustomerFacingAPI, Resource: acme-customer-pii-data, Policy: DataLakeFullAccess


### Remediation ROI (Return on Investment) Analysis

The following code snippet performs a **Remediation ROI Analysis**. While the previous use cases identified specific points of danger, this query shifts to the **Strategy** level. It identifies which development projects (Repositories) are the primary "sources" of risk across your entire production environment.

#### What this Query is Doing

This query uses the graph to calculate the "density" of risk associated with your source code management:

1. **Vulnerability Consolidation:** It groups all **CVEs** found in specific **Libraries** and traces them back to the source **Repository** (`r:Repo`) from which the software was built.
2. **Deployment Mapping:** It then looks at every **Build Artifact** generated from that repository and checks where they are currently deployed as live **Applications**.
3. **Exposure Weighting:** It filters these deployments to find **Compute Instances** that are internet-facing (`public_ip IS NOT NULL`).
4. **Aggregation:** Finally, it counts the total number of unique vulnerabilities and the number of exposed servers per repository, sorting them to show where a single "fix" (a code change or dependency update) would have the greatest impact.

#### Why This is Important

In a large organization, the security team cannot fix vulnerabilities themselves; they must ask the development teams to do it. This query is the bridge between **Security** and **DevOps**:

* **Maximum Risk Reduction:** It tells you where a single developer effort—such as updating a base image or a root dependency in a specific repository—will simultaneously patch dozens of vulnerabilities across hundreds of running servers. This is the most efficient way to "drain the swamp" of technical debt.
* **Ownership Accountability:** By linking vulnerabilities back to a **Repo**, you identify exactly which engineering team owns the fix. This moves the conversation from "We have a server with a bug" to "Team Alpha has a project that is introducing 50 high-risk exposures."
* **Infrastructure-Wide Visibility:** It reveals systemic issues. For example, if one repository shows a high count of `exposed_instances`, it suggests that this specific application is a major part of your external attack surface, making its security health a top priority for the business.

In [11]:
query = """
MATCH (v:CVE)-[:IDENTIFIED_IN]->(:Library)-[:DEPENDENCY_OF]->(:BuildArtifact)-[:BUILT_FROM]->(r:Repo)
MATCH (r)<-[:BUILT_FROM]-(:BuildArtifact)-[:RUNNING_AS]-(:Application)-[:HOSTED_ON]-(i:ComputeInstance)
WHERE i.public_ip IS NOT NULL
RETURN r.name, COUNT(DISTINCT v) AS total_vulns, COUNT(DISTINCT i) AS exposed_instances
ORDER BY total_vulns DESC
"""
results = run_cypher(query)
for record in results:
    print(f"Repository: {record['r.name']}, Total Vulnerabilities: {record['total_vulns']}, Exposed Instances: {record['exposed_instances']}")

Repository: customer-api-prod, Total Vulnerabilities: 2, Exposed Instances: 1


### Contextual Risk Scoring

The following code snippet implements a **Dynamic Risk Engine**. It represents the "final boss" of Vulnerability Prioritization: moving away from static severity numbers (CVSS) and toward a living, breathing **Contextual Risk Score** that reflects the reality of your specific environment.

#### What this Query is Doing

The query acts as a weighted calculator that adjusts the "importance" of a vulnerability based on its location and permissions:

1. **Baseline Severity:** It starts with the standard `v.baseScore` (CVSS) provided by the NVD.
2. **Reachability Weighting:** It checks if the asset is internet-facing. If a `public_ip` is present, it applies a **1.5x multiplier**. This recognizes that a vulnerability that can be touched by anyone on the internet is significantly more dangerous than one hidden behind a firewall.
3. **Impact Weighting:** It uses an `OPTIONAL MATCH` to see if the asset has an Identity that can access sensitive **Cloud Services**. If a path to data exists, it applies a **2.0x multiplier**.
4. **Compound Scoring:** It multiplies these factors together. A "Critical" 10.0 vulnerability on a public-facing server with admin access to S3 would result in a **Final Risk Score of 30.0** (), while the same 10.0 on an isolated server stays at **10.0**.

#### Why This is Important

This approach solves the single biggest problem in cybersecurity: **Alert Fatigue.** By calculating risk this way, you gain several strategic advantages:

* **True Prioritization:** In a traditional list, you might have 500 "Critical" vulnerabilities. This query re-orders that list so the ones that actually lead to a data breach float to the very top. It tells the team exactly what to fix in the first hour of their shift.
* **Business Alignment:** It translates technical debt into business risk. By factoring in `CloudService` access, you are essentially telling stakeholders, *"We aren't just patching a server; we are protecting our Customer Database."*
* **Dynamic Defense:** As your infrastructure changes (e.g., a developer opens a port or attaches a new IAM policy), the graph automatically recalculates these scores. Your prioritization is always based on the **current** state of the network, not a snapshot from last month's scan.

In [12]:
query = """
MATCH (v:CVE)-[:IDENTIFIED_IN]->(:Library)-[:DEPENDENCY_OF]->(:BuildArtifact)-[:RUNNING_AS]->(app:Application)-[:HOSTED_ON]->(ins:ComputeInstance)

// Calculate Reachability
WITH v, app, ins, 
     CASE WHEN ins.public_ip IS NOT NULL THEN 1.5 ELSE 1.0 END AS reachability_multiplier

// Calculate Impact (Passing reachability_multiplier forward)
OPTIONAL MATCH (ins)-[:RUNS_AS]->(:Identity)-[:ASSUMES]->(:IAMPolicy)-[:HAS_ACCESS_TO]->(cs:CloudService)
WITH v, app, ins, reachability_multiplier,
     CASE WHEN cs IS NOT NULL THEN 2.0 ELSE 1.0 END AS impact_multiplier

// Final Calculation
WITH v, app, ins, reachability_multiplier, impact_multiplier,
     (v.baseScore * reachability_multiplier * impact_multiplier) AS contextual_risk_score
     
RETURN v.id AS CVE,
       app.name AS Application,
       v.baseScore AS Base_CVSS,
       reachability_multiplier AS Reach_Mult,
       impact_multiplier AS Impact_Mult,
       round(contextual_risk_score, 2) AS Final_Risk_Score
ORDER BY Final_Risk_Score DESC
"""
results = run_cypher(query)
for record in results:
    print(f"CVE: {record['CVE']}, Application: {record['Application']}, Base CVSS: {record['Base_CVSS']}, Reach Mult: {record['Reach_Mult']}, Impact Mult: {record['Impact_Mult']}, Final Risk Score: {record['Final_Risk_Score']}")

CVE: CVE-2021-44228, Application: CustomerFacingAPI, Base CVSS: 10, Reach Mult: 1.5, Impact Mult: 2.0, Final Risk Score: 30.0
CVE: CVE-2021-45046, Application: CustomerFacingAPI, Base CVSS: 9, Reach Mult: 1.5, Impact Mult: 2.0, Final Risk Score: 27.0
CVE: CVE-2021-44228, Application: LegacyParser, Base CVSS: 10, Reach Mult: 1.0, Impact Mult: 1.0, Final Risk Score: 10.0
CVE: CVE-2021-45046, Application: LegacyParser, Base CVSS: 9, Reach Mult: 1.0, Impact Mult: 1.0, Final Risk Score: 9.0
